In [1]:
import numpy as np
import graphviz
import pandas as pd
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

In [2]:
class DataAnalyzer:
    def __init__(self, data_path):
        self.data_path = data_path
        self.data = pd.read_csv(data_path)

    def value_counts(self, column_name):
        return self.data[column_name].value_counts()

    def entropy(self, column):
        counts = Counter(column)
        probabilities = [count / len(column) for count in counts.values()]
        return -sum(p * np.log2(p) for p in probabilities if p > 0)

    def average_entropy(self, attribute, target):
        values = self.data[attribute].unique()
        average_entropy = 0
        for value in values:
            subset = self.data[self.data[attribute] == value]
            avg_entropy = self.entropy(subset[target].values)
            average_entropy += (len(subset) / len(self.data)) * avg_entropy
        return average_entropy

    def information_gain(self, attribute, target):
        target_entropy = self.entropy(self.data[target].values)
        avg_entropy = self.average_entropy(attribute, target)
        return target_entropy - avg_entropy

In [3]:
class DecisionTreeModel:
    def __init__(self, data_path):
        self.data_path = data_path
        self.model = DecisionTreeClassifier(criterion='entropy')
        self.load_data()

    def load_data(self):
        # Tải dữ liệu từ file CSV
        self.data = pd.read_csv(self.data_path)
        # Kiểm tra giá trị khuyết
        self.missing_values = self.data.isnull().sum()
        # Phân tách dữ liệu
        features = self.data.columns
        pattern = re.compile(r'^Q\d+$')
        features = [feature for feature in features if pattern.match(feature)]
        self.X = self.data[features]
        self.y = self.data['Rank']

    def split_data(self, test_size=0.2):
        # Phân chia dữ liệu thành tập huấn luyện và tập kiểm tra
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=test_size)

    def train_model(self):
        # Huấn luyện mô hình
        self.model.fit(self.X_train, self.y_train)

    def evaluate_model(self):
        # Dự đoán trên tập kiểm tra
        self.y_pred = self.model.predict(self.X_test)
        # Đánh giá mô hình
        self.accuracy = accuracy_score(self.y_test, self.y_pred)
        '''self.classification_report = classification_report(self.y_test, self.y_pred)'''
        print("Độ chính xác là:", self.accuracy)
        ''' print("Classification Report:\n", self.classification_report)'''

    def plot_tree(self):
        # Vẽ biểu đồ cây quyết định
        plt.figure(figsize=(160,80))
        features = self.data.columns
        pattern = re.compile(r'^Q\d+$')
        features = [feature for feature in features if pattern.match(feature)]
        plot_tree(self.model, filled=True, feature_names=features, class_names=True)
        plt.show()

In [4]:


# Sử dụng lớp đã tạo
if __name__ == '__main__':

    data_path = '/content/dt_data.csv'
    analyzer = DataAnalyzer(data_path)

    attribute = input("Nhập tên của thuộc tính điểm số (Q1 - Q9): ")
    target = 'Rank'

    # Hiển thị các giá trị tính toán
    print(f"Entropy (H) của thuộc tính {attribute}: {analyzer.entropy(analyzer.data[attribute].values)}")
    print(f"Average Entropy (AE) của thuộc tính {attribute}: {analyzer.average_entropy(attribute, target)}")
    print(f"Information Gain (IG) của thuộc tính {attribute}: {analyzer.information_gain(attribute, target)}")

    tree_model = DecisionTreeModel('/content/dt_data.csv')
    tree_model.split_data(test_size=0.2)
    tree_model.train_model()
    tree_model.evaluate_model()
    tree_model.plot_tree()




Nhập tên của thuộc tính điểm số (Q1 - Q9): Q2
Entropy (H) của thuộc tính Q2: 4.132418142909147
Average Entropy (AE) của thuộc tính Q2: 0.7616877890146895
Information Gain (IG) của thuộc tính Q2: 0.1603079140797511
Độ chính xác là: 0.6851851851851852
